# Solutions · 09 · Waves, strings and beams

Worked solutions to the exercises of [notebook 09](../notebooks/09_waves_strings_and_beams.ipynb). Try each exercise
yourself before reading its solution - the learning happens in the attempt. Solutions are one way to
answer each question; other correct approaches exist.

In [1]:
try:                      # on Google Colab (or anywhere engmolines is missing): install it from GitHub
    import engmolines
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ktwyw/engmolines"], check=True)
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from engmolines import problems as pr, integrators as ti, analysis, operators as op, boundary as bc
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 90, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})

import inspect
from IPython.display import Code

def show_source(obj):
    """Display the source code of a library function or class."""
    return Code(inspect.getsource(obj), language="python")

def heading(text):
    print(f"\n{text}\n" + "=" * len(text))

## Exercise 1

Integrate the string with the trapezoidal rule (A-stable, non-dissipative) for 20 periods: is the energy conserved? Is the phase correct? Compare with Verlet at the same step.

In [2]:
p = pr.wave_string(mode=2); n = 101; T = p.t_end
tz = ti.trapezoidal(p.rhs, p.y0, (0, 20 * T), 0.005, band=(n + 1, n + 1), n_out=20)
sv = ti.verlet(p.params["accel"], p.y0[:n], p.y0[n:], (0, 20 * T), 0.005, n_out=20)
for name, s in (("trapezoidal", tz), ("Verlet", sv)):
    E = np.array([analysis.wave_energy(y[:n], y[n:], 1.0, p.grid) for y in s.y])
    print(f"{name:<12}: energy change over 20 periods {E[-1] / E[0] - 1:+.2e}; position error {np.max(np.abs(s.y[-1][:n] - p.exact(p.x, s.t[-1]))):.2e}")

trapezoidal : energy change over 20 periods +9.49e-07; position error 4.80e-04
Verlet      : energy change over 20 periods +1.78e-07; position error 1.20e-04


The trapezoidal rule has |R(iy)| = 1 exactly: it conserves the energy of a linear wave to rounding, like
Verlet. Both make a phase error of order (hω)², so their position errors after twenty periods are similar;
the trapezoidal rule is implicit and A-stable (any step is stable, if inaccurate) while Verlet is explicit
with the limit hω ≤ 2 - for undamped linear waves either is a sound choice, and BDF is not.

## Exercise 2

Give the beam a clamped left end and a free right end (`biharmonic(w, g, 'clamped', 'free')`) and find its first natural frequency from a long integration (the FFT of the tip displacement); compare with (1.875)^2 sqrt(EI/rhoA)/L^2.

In [3]:
from engmolines import grids
L, n = 1.0, 60
g = grids.uniform(0, L, n)
accel = lambda t, w: -op.biharmonic(w, g, "clamped", "free")   # noqa: E731
w0 = (g.x / L) ** 2 * (3 - 2 * g.x / L) * 0.01                 # a smooth clamped-free deflection as the start
s = ti.verlet(accel, w0, np.zeros(n + 1), (0, 20.0), 0.4 * g.dx**2, n_out=4000)
tip = s.y[:, n]
freqs = np.fft.rfftfreq(len(tip), s.t[1] - s.t[0])
spec = np.abs(np.fft.rfft(tip - tip.mean()))
f1 = freqs[np.argmax(spec[1:]) + 1]
print(f"first natural frequency from the tip's spectrum: {f1:.4f} Hz; theory (1.875)^2 sqrt(EI/rhoA)/(2 pi L^2) = {1.8751**2 / (2 * np.pi):.4f} Hz")

first natural frequency from the tip's spectrum: 0.5499 Hz; theory (1.875)^2 sqrt(EI/rhoA)/(2 pi L^2) = 0.5596 Hz


A clamped-free beam's tip displacement, Fourier-transformed, peaks at the first natural frequency
(βL)²√(EI/ρA)/(2πL²) with βL = 1.875 - within the resolution of a 20-time-unit record. The `biharmonic`
operator's free-end ghosts impose w″ = w‴ = 0 at the tip.

## Exercise 3

**Implement it yourself:** write the energy-conserving Verlet step for the wave equation with a damping term -2 a v (treat the damping implicitly in the velocity update) and verify the decay rate against the telegraph solution.

In [4]:
p = pr.telegraph(a=0.5); n = 101; h = 0.005
u, v = p.y0[:n].copy(), p.y0[n:].copy()
accel = lambda t, w: op.diffusion(w, p.grid, 1.0, bc.Dirichlet(0.0), bc.Dirichlet(0.0), t)   # noqa: E731
a = 0.5
acc = accel(0, u)
for k in range(int(round(3.0 / h))):
    v_half = (v + 0.5 * h * acc) / (1 + a * h)             # damping -2 a v treated implicitly over the half step
    u = u + h * v_half
    acc = accel(0, u)
    v = (v_half + 0.5 * h * acc) / (1 + a * h)
    u[0] = u[-1] = 0.0; v[0] = v[-1] = 0.0
print(f"damped Verlet: error vs e^(-at) sin(pi x) cos(omega t) = {np.max(np.abs(u - p.exact(p.x, 3.0))):.2e} (the amplitude has decayed by e^(-3a) = {np.exp(-1.5):.4f})")

damped Verlet: error vs e^(-at) sin(pi x) cos(omega t) = 2.58e-04 (the amplitude has decayed by e^(-3a) = 0.2231)


Treating the damping implicitly in each velocity half-update keeps the scheme explicit in the stiff wave
operator and unconditionally stable in the damping, and preserves second order; the decay rate e^(−at) of
the telegraph solution is reproduced.